---
title: "Module 2 - Lab 2"
subtitle: "Sparse vs Dense Semantic Representations for SEC Chunks"
number-sections: true
date: "2026-02-01"
date-modified: today
date-format: long
engine: jupyter
format:
  html:
    toc: true
    toc-depth: 3
    code-overflow: wrap
execute:
  echo: true
  eval: false
  warning: false
  message: false
categories: ['M02:', 'Lab']
description: "Hands-on lab for Module 2 focused on comparing sparse and dense semantic representations over SEC chunk data."
---

## Overview

In this lab you will compare **sparse** and **dense** text representations on chunked SEC filing text, then connect those results to classification, probability, and error analysis.

Starting from a chunked corpus, you will:

1. inspect the chunk metadata and text distribution
2. build a TF-IDF baseline
3. build dense sentence embeddings
4. compare retrieval results for the same business query
5. train a simple classifier
6. interpret cross-entropy / log loss
7. inspect high-loss examples and model failures

## How This Lab Connects

- `M02_T1` and `M02_Lab1` produce the kind of chunk-level corpus used here.
- `M02_T2` demonstrates the core semantic retrieval and cross-entropy ideas in a guided tutorial.
- This lab makes you execute those steps yourself with validation and interpretation.
- `M02_A` then moves from general dense representations to a more specific assignment workflow using rival-company filings, Word2Vec, GloVe, and embedding geometry.

::: {.callout-important}
The tutorials are meant to help with mechanics and interpretation. The lab requires you to adapt the workflow and show your own executed evidence rather than copying a single canned path.
:::

## Recommended Notebook Pattern

Use the following structure:

1. Setup
2. Data inspection
3. Sparse baseline
4. Dense embedding workflow
5. Supervised classification
6. Cross-entropy interpretation
7. Error analysis
8. Reflection

## Validation Standard

Students should show intermediate evidence, not only final outputs. Typical checks include:

- row counts and example chunks
- matrix shapes
- top retrieval results
- label distribution
- classification metrics
- log loss values
- examples of high-loss or misclassified rows

## Setup

In [ ]:
#| eval: false
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

## Load a Chunked SEC Corpus

Use either the Assignment 1 chunk file or the lab-scale chunk file you created in `M02_Lab1`.

In [ ]:
#| eval: false
candidate_paths = [
    Path("../data/assignment01/chunks.csv"),
    Path("../data/outputs/module2_lab1/lab1_chunks.csv"),
]

chunks_path = next((path for path in candidate_paths if path.exists()), None)
if chunks_path is None:
    raise FileNotFoundError("Could not find an SEC chunk file. Run Lab 1 first or provide assignment01/chunks.csv.")

chunks_df = pd.read_csv(chunks_path)
print("Using:", chunks_path)
print(chunks_df.shape)
chunks_df.head()

In [ ]:
#| eval: false
required_cols = [col for col in ["chunk_id", "company", "item", "chunk_text", "char_count"] if col in chunks_df.columns]
chunks_df = chunks_df[required_cols].dropna(subset=["chunk_text"]).copy()

if "company" not in chunks_df.columns:
    chunks_df["company"] = "sample_corpus"
if "item" not in chunks_df.columns:
    chunks_df["item"] = "unknown_item"
if "char_count" not in chunks_df.columns:
    chunks_df["char_count"] = chunks_df["chunk_text"].str.len()

chunks_df = chunks_df.reset_index(drop=True)
print(chunks_df.shape)
chunks_df.head()

## Inspect the Data

In [ ]:
#| eval: false
print(chunks_df["company"].value_counts(dropna=False))
print(chunks_df["item"].value_counts(dropna=False))
chunks_df["char_count"].describe()

In [ ]:
#| eval: false
chunks_df["char_count"].plot(kind="hist", bins=30, figsize=(7, 4), title="Chunk Length Distribution")
plt.xlabel("Characters")
plt.show()

## Build a Sparse TF-IDF Baseline

In [ ]:
#| eval: false
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(chunks_df["chunk_text"])
print(X_tfidf.shape)

Use a business-oriented query so retrieval is easy to interpret.

In [ ]:
#| eval: false
query = "cybersecurity risk and customer data privacy issues"
q_tfidf = tfidf.transform([query])
scores_tfidf = cosine_similarity(q_tfidf, X_tfidf)[0]

top_sparse = chunks_df.iloc[np.argsort(scores_tfidf)[-5:][::-1]].copy()
top_sparse["tfidf_score"] = np.sort(scores_tfidf)[-5:][::-1]
top_sparse[["chunk_id", "company", "item", "tfidf_score", "chunk_text"]]

## Build Dense Embeddings

In [ ]:
#| eval: false
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
X_embed = embed_model.encode(
    chunks_df["chunk_text"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(X_embed.shape)

In [ ]:
#| eval: false
q_embed = embed_model.encode([query], normalize_embeddings=True)
scores_embed = cosine_similarity(q_embed, X_embed)[0]

top_dense = chunks_df.iloc[np.argsort(scores_embed)[-5:][::-1]].copy()
top_dense["embed_score"] = np.sort(scores_embed)[-5:][::-1]
top_dense[["chunk_id", "company", "item", "embed_score", "chunk_text"]]

## Compare Sparse and Dense Retrieval

In [ ]:
#| eval: false
comparison = pd.DataFrame({
    "sparse_chunk_id": top_sparse["chunk_id"].tolist(),
    "dense_chunk_id": top_dense["chunk_id"].tolist(),
    "sparse_item": top_sparse["item"].tolist(),
    "dense_item": top_dense["item"].tolist(),
})
comparison

Write 2 to 3 sentences interpreting:

- where TF-IDF benefits from lexical overlap
- where dense retrieval finds semantically related chunks with different wording
- where dense retrieval seems too broad or generic

## Create a Weak Label for Classification

This is intentionally simple so you can focus on representation and diagnostics.

In [ ]:
#| eval: false
chunks_df["label"] = chunks_df["chunk_text"].str.contains(
    r"privacy|cyber|security|data breach|information security",
    case=False,
    regex=True,
).astype(int)

chunks_df["label"].value_counts(dropna=False)

## Train a TF-IDF Classifier

In [ ]:
#| eval: false
indices = np.arange(len(chunks_df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=chunks_df["label"],
)

X_train_tfidf = X_tfidf[train_idx]
X_test_tfidf = X_tfidf[test_idx]
y_train = chunks_df.loc[train_idx, "label"]
y_test = chunks_df.loc[test_idx, "label"]

clf_tfidf = LogisticRegression(max_iter=300)
clf_tfidf.fit(X_train_tfidf, y_train)
probs_tfidf = clf_tfidf.predict_proba(X_test_tfidf)
preds_tfidf = clf_tfidf.predict(X_test_tfidf)

print(classification_report(y_test, preds_tfidf))

## Train a Dense-Embedding Classifier

In [ ]:
#| eval: false
X_train_emb = X_embed[train_idx]
X_test_emb = X_embed[test_idx]

clf_embed = LogisticRegression(max_iter=300)
clf_embed.fit(X_train_emb, y_train)
probs_embed = clf_embed.predict_proba(X_test_emb)
preds_embed = clf_embed.predict(X_test_emb)

print(classification_report(y_test, preds_embed))

## Compare Metrics and Confusion Matrices

In [ ]:
#| eval: false
metrics_df = pd.DataFrame({
    "model": ["TF-IDF + Logistic Regression", "Dense Embeddings + Logistic Regression"],
    "log_loss": [
        log_loss(y_test, probs_tfidf),
        log_loss(y_test, probs_embed),
    ],
})
metrics_df

In [ ]:
#| eval: false
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ConfusionMatrixDisplay(confusion_matrix(y_test, preds_tfidf)).plot(ax=axes[0], colorbar=False)
axes[0].set_title("TF-IDF")

ConfusionMatrixDisplay(confusion_matrix(y_test, preds_embed)).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Dense Embeddings")

plt.tight_layout()
plt.show()

## Connect to Cross-Entropy

In [ ]:
#| eval: false
print("TF-IDF log loss:", round(log_loss(y_test, probs_tfidf), 4))
print("Dense log loss:", round(log_loss(y_test, probs_embed), 4))

Interpretation prompts:

- What does a **high-loss** example mean for one row?
- Why are confident mistakes punished more heavily?
- How is this related to next-token prediction in modern language models?

## Inspect High-Loss Examples

In [ ]:
#| eval: false
test_rows = chunks_df.iloc[test_idx].copy()
test_rows["p_positive_tfidf"] = probs_tfidf[:, 1]
test_rows["loss_tfidf"] = -(
    y_test.values * np.log(np.clip(probs_tfidf[:, 1], 1e-9, 1.0)) +
    (1 - y_test.values) * np.log(np.clip(probs_tfidf[:, 0], 1e-9, 1.0))
)

test_rows.sort_values("loss_tfidf", ascending=False)[
    ["chunk_id", "company", "item", "label", "p_positive_tfidf", "loss_tfidf", "chunk_text"]
].head(10)

For at least five examples, explain whether the apparent failure is driven by:

- ambiguous language
- weak labels
- missing context
- lexical mismatch
- generic business wording

## Reflection

Answer all of the following:

1. When did sparse features outperform expectations?
2. When did dense embeddings provide clearer business value?
3. Why is cross-entropy a natural training objective for text models?
4. How does this lab help explain what an LLM is doing during training?

## Optional Stretch Work

Choose one if you want to go further:

1. compare two embedding models
2. cluster dense embeddings and interpret the themes
3. compare ROC-AUC and PR-AUC under class imbalance
4. replace sentence embeddings with mean-pooled Word2Vec or GloVe vectors to preview the assignment

## AI Use and Share Links {.unnumbered}

If generative AI materially supports your work for this lab, include an AI disclosure appendix or separate AI disclosure document if your instructor requests one. Include the complete prompt(s), relevant output excerpt(s), validation steps, and direct shared chat link(s) when available.

![](../M0/M0_lecture01_figures/chatgpt-share.jpg){width="80%" fig-align="center"}

![](../M0/M0_lecture01_figures/gemini-share-conversation.png){width="80%" fig-align="center"}

When possible, use **one continuous chat** for the activity so the full reasoning trail can be reviewed in one place. If you used multiple chats, share **all chats directly related** to the work.

Tools such as **ChatGPT**, **Claude**, and **Gemini** support direct chat sharing. If sharing or export is not available, include copied prompt/output evidence or screenshots instead. Because shared links may be viewable by anyone with the link, do not include confidential, personal, or restricted information in those chats.